In [ ]:
from pathlib import Path
import sys, subprocess, hashlib, json, warnings, time
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from IPython.display import display

# Run with an environment containing the scRITMO dependencies.
LOCAL_REPO = Path('external/scRITMO')
RUN_DIR = Path('outputs/simulation/output/scRITMO_extended_simulation/sigma_gradient_1p5_2p5_3p5_4p5_5p5_6p5_empirical_fixed_v1')
H5AD = RUN_DIR / 'scRITMO_extended_simulation.h5ad'
OUTPUT_DIR = RUN_DIR / 'R_input' / 'cell_phase_benchmark_scRITMO'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 1
N_EPOCHS = 300
N_THETA_POST = 100


In [ ]:
OFFICIAL_COMMIT = '2285bb45f21f57a3498ac92a4c5401429070dcbd'
SOURCE_SHA256 = {'README.md': '901dd7d44a8f27f1f60bc00a573a12dd047ba7536510b7c0aa0237a93e764b96', 'tutorial_inference.ipynb': 'ced01c70d152006268cb07e67c46fe65dcf71c486c0f39a64f37ec0da739fb58', 'ccg_zhang_context.csv': '395f3e37cf4034d319266018633f80307e66b83f79cc38bcea3562380f164182', 'src/scritmo/glm.py': '68ff9998afe3cf31af3168fb74b9cf81947e23b58878e742ba6f11e5b87ae92a', 'src/scritmo/pseudobulk.py': 'ad1cc52d5556df88b5656f54ea2bcd2f7cf0f6cd3c6f89fbac13229ecd299c58', 'src/scritmo/beta.py': 'e1ddf2e5afa5857962c66332e02f4279f5ee35880fef9f5240b9102a40991878', 'src/scritmo/__init__.py': '06566b85470ef97b875cf10e02defcd20c01339dfe9a514dfe1916832389eb2f', 'src/scritmo/power_spherical.py': '9c0af76e678c50aa145ed2578c4480350ea7616a9f64c10f73468dba1507c448', 'src/scritmo/gene_lists.py': 'c5a5809dc982b914dfa59925ce2cc30e90e2fa4d0dabaae77e96465199e3dd70', 'src/scritmo/basics.py': 'df8541d9a1eefeac6a6f277d892d3c779a8c3151df89d05c8f75ee54d477dec8', 'src/scritmo/linear_regression.py': 'a37bdd40c1eb499a3430519011e7e9dc4f70ce0dd25a49f4d5162b8652d1391f', 'src/scritmo/plot/plot_generator_fns.py': '72edf4e54a8f7a423bf126b40951966c8cadf0169cb6d913793dfb818e7f9521', 'src/scritmo/plot/bar_box.py': 'b108a05a594ca74a21a81d877d90032975ab00f765e24f7a078422f04cb4c460', 'src/scritmo/plot/misc.py': 'cd988c3cf5ddd1bebf7bbbfd800ab792c04e4438627e4b5d78a86b7268249331', 'src/scritmo/plot/panels_plot_helpers.py': 'f401dab2d6c94df2aaf087e352cb8581af5ca54c9fdb12ea60dc2cd38cf450a8', 'src/scritmo/plot/panels_constants.py': '71411b67be587cc7253aa39589bb2d46217a58a599e3289a2adf09e4eaea0327', 'src/scritmo/plot/__init__.py': '7afb6eecb7e38b5c8ce8c8304c8c963dc462ab637a3c08de71afacdab628146f', 'src/scritmo/plot/cedric_fn.py': 'b3e50eced83c8641af9b820f1773f25974523d6a5bba5205fb4e5e5b60e5dd85', 'src/scritmo/plot/utils.py': '781054ed9f5575de392b64fe4a48fadc2ea86b49e585a729a2659c3912a0d2bd', 'src/scritmo/plot/data_and_fits.py': 'c687e5b155ae5e42306f5ab7bda71e300e62436824d00bb31ee24f768db4abf2', 'src/scritmo/plot/histos.py': '01271fefd2cca71a1858f667425412467f18597f19e7fc716c8e9ea9e826a5c2', 'src/scritmo/dryseq/dryseq_main.py': '1590dcb187c7a521c4a501c5b93bf76042c25475374ec18f477dcced7a0bde98', 'src/scritmo/dryseq/dryseq_fitting.py': 'b0bc204c39865f68145043187f8e0420a470e19ec3137a93466ac8e23df13ca9', 'src/scritmo/dryseq/dryseq_utils.py': 'd54a2fceb1018086f3806870cd1935a6dfa114b927dc449dc096373d88765ef9', 'src/scritmo/ml/extra_genes.py': 'e676b2a25206003a966cfc948a352b3906b09e63e55fbec553aa15e2ec3d8198', 'src/scritmo/ml/context_model.py': 'f9e363464b37b43b9e97eaf924ae0b1bf169576c471086659cece7235fdd5893', 'src/scritmo/ml/__init__.py': 'cdb8bf834cdc3e41487b04cdb2700b7362edc5574cd657185c161cefdfeb6cf9', 'src/scritmo/ml/marginalization.py': '83d73bcedddb1533e65eb18ad8f79e21dfdba7a8ffcdb073ca9e0e4357272f53', 'src/scritmo/ml/warmup.py': 'fd0f77428c414cd0643ca4e4e0ae432d9ef654d43a6d2c22c3c6d65c8e526b15', 'src/scritmo/ml/genome_fit.py': 'aa00bc59ccdf5ef1e461cf898ce255cdfd1a1765929bd907499514ecf1943b9d', 'src/scritmo/ml/utils.py': '9846adfafe779ab39a80890a962e8dc4f6c649754e09baaa675c016b23ff5e93', 'src/scritmo/ml/discrete_MI.py': '6275ce8f4d63b341fdb52e90c13c6c9335e9aa0d6e6d8bd7175f835a55885931', 'src/scritmo/ml/ensemble.py': '85d06c073a243cc2b04eaabbf7fc75fbb3b2f3db01603ee81a118cefb795e4c7', 'src/scritmo/ml/trainer.py': '1b4559b6f3d482c717f6521d36511b29fee34728a0ac2d4a22483a9485dbf530', 'src/scritmo/ml/analysis_utils.py': '1c729164fd2d3b130cfdb495f31b33e07fdfedf80a556af18873f4e68d4437fa', 'src/scritmo/ml/sigma_marginal.py': 'c3cc5c4d30985d90b290df5faeed3be23d71d15ee260c5be76925ead9a3e89be', 'src/scritmo/ml/desync_mixin.py': 'db2caaa13f2aae4cce89014462b5e3a15a5cbb16089bdbda71508f0af3f4028b', 'src/scritmo/ml/null_model.py': 'df21406cf8d2f90923be24670db7ade0819efee98f1f251c42f166423cc09e5b', 'src/scritmo/ml/misc/gene_search.py': '069086b182511479e85d22894a1b6af547809d857c9900375a4112e49de8149a', 'src/scritmo/ml/misc/gene_optim.py': '0048b4c779875927ce866522a91fc6a6cf4aa5325e01bd91eb8f474b0010f62c', 'src/scritmo/ml/misc/tempo_wrappers.py': '80121dec21710632212c674baa0ad1ff9bb0f9ae51db741ee1f9af6bdcb9035b', 'src/scritmo/ml/misc/pseudo_run.py': 'f4c6d183532ec57505713a96256095f032e25331e5ef486ee79db70c38dc898d', 'src/scritmo/ml/misc/svi/model_guide.py': '811823ddb21f5a0f544433a785266676b1a4289688e2b000ab739bc9cdc44fc6', 'src/scritmo/ml/misc/svi/svi.py': '6f826b712491df9c36116bb17c121066bd761398600c779216007f1e10052ff3', 'src/scritmo/ml/misc/power_spherical/torch_distribution.py': 'b09d6cf55596a28ba447957bced0bd5ecdb7eb7f4d172d296f8b2b67a9c44e47', 'src/scritmo/ml/misc/power_spherical/power_spherical.py': '893f9d92259f0a452518abfdc17114dc5981bea847be83232dd681b7821ab02c', 'src/scritmo/ml/simulations/old_fn.py': '0e716a38a42c3a41a1d36daf333c6b7f44ce2ba72bbaa1a5580a1ebb360117b1', 'src/scritmo/ml/simulations/plot.py': 'cea43467d1ac5cdad31dc9c17affb4f5ee3e09fa7dc995799e11cb1a671e80b4', 'src/scritmo/ml/simulations/simulation_plot.py': '66a99f951e305172010e0601470dcdd4e4bbd6eaa03b97a9da5ce5a8834c036d', 'src/scritmo/ml/simulations/simulations.py': 'e830d6582f3b861c7435b0ee35552cfa813bba0a4ebf1f3df636da880f642000', 'src/scritmo/ml/simulations/estimate_desynchrony.py': '2cd87109e48e6830fef90d6bdfb4bdf59fc516279022e98003ae42bf750746c8', 'src/scritmo/ml/simulations/simulate_populations.py': 'b59e3860cc1309e04eecc0243e84e948f06e4737d0da6b1f7d232e0d23ac5966', 'src/scritmo/ml/simulations/utils.py': '9e356d594b08ebcfb044aad11a6168fbe0c095c5dcc34838efe1ab063acb8805', 'src/scritmo/ml/simulations/pipeline.py': '988c6bb345aabbec92a67e3164cc4dccfe43b8deb4e8b67d9b53c7534b52fc83', 'src/scritmo/ml/unspliced/unspliced_luuk.py': '2b96bdabc283a3fe595a937ee3cb8cce06c77d513fce869690828c8d2fabfe4a', 'src/scritmo/ml/unspliced/fisher.py': 'f9fc43bd03a56f87010ee4b31b87290285bfa1d43ace98af2da6d17d06ca0b16', 'src/scritmo/ml/unspliced/unspliced.py': '2b5cbb91a7aff26b899d13f4cbe6ceac2eea9bf7abbef8b86d3128e8bc522af2', 'src/scritmo/ml/unspliced/unspliced_deg.py': 'cacc5eee3fc78ea6c0855bb499e182dfe625d63d9449c6bc9708db356c8c770c', 'src/scritmo/ml/unspliced/unspliced_0.py': '024faffdab17d38b44ed60c01ab2ca000e1cb5bc9411b0127f363e517f5e4c89', 'src/scritmo/circular/align.py': 'bc2177995400fbfd66128c1893e470afb960856b2de686c04422ef4b07934eaa', 'src/scritmo/circular/__init__.py': '2248e0f1803c6e3e7859805540d973cdd271cd2567c158c08de2653d38cc1097', 'src/scritmo/circular/von_mises.py': 'd7eb64cd813cee5ea350b24313deb7cd39fb67018a59c4ba92de1efc35790340', 'src/scritmo/circular/circular.py': '322fe4e01a80de547413c1d99413be329c5a6abb6a6ab8710dec250f52757ef5', 'src/scritmo/circular/median.py': 'd5eb4e20a15f4f51b451dc22f6b80b704ef7fbe487efee901b28e8c34b381953', 'src/scritmo/pychiral/chiral.py': '01c4e5e2f5137948e214ae543d28164adb55984f13cc61d606ef309b4735c7a6', 'src/scritmo/pychiral/helper_fn.py': '5ad08d6918f533965d2b288a84e0b16b235f83e18ab4ed75425d3035f967c070', 'src/scritmo/pychiral/__init__.py': '0f58bd01c45d402f86b931e64a891e35e089fe2540bd50c857f2f85a59b4f35d', 'src/scritmo/pychiral/em.py': '168e00f1c348e5b42fbcbdba8fcc48999f173d59174170d84806cfaaa8f1d505', 'src/scritmo/pychiral/stat_phys.py': 'be7132fd09a32ee256137c49cd3529f7a0300489ea379303e80e3fd6df902dc1'}
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def matches(repo):
    return all((repo / f).is_file() and sha256(repo / f) == h for f, h in SOURCE_SHA256.items())

# Use the configured local checkout when it matches the audited official source.
# Otherwise use an isolated pinned checkout; never reset/pull/edit LOCAL_REPO.
REPO = LOCAL_REPO
if not matches(REPO):
    REPO = OUTPUT_DIR / '_official_source' / OFFICIAL_COMMIT
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', 'https://github.com/AndreaSalati/scRITMO.git', str(REPO)], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', OFFICIAL_COMMIT], check=True)
    assert matches(REPO), 'Cached official source differs; inspect it before rerunning.'
assert not any(x == 'scritmo' or x.startswith('scritmo.') for x in sys.modules), 'Restart kernel before Run All.'
sys.path.insert(0, str(REPO / 'src'))
import scritmo as sr
import scritmo.ml as cr
import torch
assert Path(sr.__file__).resolve().is_relative_to(REPO.resolve())
torch.set_num_threads(2)
torch.manual_seed(SEED)
np.random.seed(SEED)
print('Official source:', REPO, '\nCommit:', OFFICIAL_COMMIT)


In [ ]:
assert H5AD.is_file(), H5AD
adata = ad.read_h5ad(H5AD)
assert adata.shape == (9000, 1000), adata.shape
assert adata.obs_names.is_unique and adata.var_names.is_unique
assert all(str(x).strip() for x in adata.obs_names)
adata.X = sparse.csr_matrix(adata.X)
v = adata.X.data
assert np.isfinite(v).all() and (v >= 0).all() and np.equal(v, np.floor(v)).all()
meta = adata.obs.rename(columns={'true_phase_hour':'T_true', 'true_phase_rad':'Theta_true'}).copy()
meta.insert(0, 'cell', adata.obs_names.astype(str))
required = ['cell','sample_name','ZT','celltype','sigma_bio_hour','T_true','Theta_true']
meta = meta[required].copy()
assert not meta.isna().any().any()
assert np.isfinite(meta[['ZT','sigma_bio_hour','T_true','Theta_true']].to_numpy()).all()
assert set(meta.ZT) == {0,4,8,12,16,20}
expected_sigma = {f'CT{i}': s for i,s in enumerate([1.5,2.5,3.5,4.5,5.5,6.5],1)}
assert set(meta.celltype) == set(expected_sigma)
assert np.allclose(meta.sigma_bio_hour, meta.celltype.map(expected_sigma).astype(float))
assert (meta.groupby('sample_name').size() == 500).all()
assert (meta.groupby('sample_name').ZT.nunique() == 1).all()
assert (meta[['sample_name','ZT']].drop_duplicates().groupby('ZT').size() == 3).all()
assert ((meta.T_true >= 0) & (meta.T_true < 24)).all()
assert np.allclose(((meta.Theta_true*24/(2*np.pi)-meta.T_true+12)%24)-12,0,atol=1e-8)
config = json.loads(adata.uns['config_json'])
assert config['perturbations_path'] is None
assert all(x['phase_offset_hour'] == 0 for x in config['celltypes'].values())

rmeta = pd.read_csv(RUN_DIR / 'R_input' / 'metadata.csv')
assert rmeta.cell.is_unique and set(rmeta.cell) == set(meta.cell)
rmeta = rmeta.set_index('cell').loc[meta.cell]
for col in required[1:]:
    if pd.api.types.is_numeric_dtype(meta[col]):
        assert np.allclose(meta[col], rmeta[col])
    else:
        assert np.array_equal(meta[col].astype(str), rmeta[col].astype(str))

clock_file = REPO / 'ccg_zhang_context.csv'
params_g = sr.Beta(str(clock_file))
assert len(params_g) == 15 and params_g.index.is_unique
reference_qc = pd.DataFrame({'gene': params_g.index, 'present': params_g.index.isin(adata.var_names)})
assert reference_qc.present.all(), reference_qc
assert np.isfinite(params_g[['a_0','a_1','b_1','amp','phase','disp']]).all().all()
assert (params_g.disp > 0).all()
reference_qc.to_csv(OUTPUT_DIR / 'reference_gene_QC.csv', index=False)
counts = adata.obs['library_size'].to_numpy(dtype=float)
assert counts.shape == (9000,) and np.isfinite(counts).all() and (counts > 0).all()
# Remove truth, sample times, celltype and sigma from the model input.
model_data = ad.AnnData(X=adata.X.copy(), obs=pd.DataFrame(index=adata.obs_names.copy()),
                       var=pd.DataFrame(index=adata.var_names.copy()))
assert model_data.shape == (9000,1000)
print('Input cells × genes:', model_data.shape, '| References:', len(params_g))
print('Cell alignment: passed; counts: raw nonnegative integers; offset: full-library exposure')
display(reference_qc)
display(meta.groupby(['celltype','sigma_bio_hour'], observed=True).size().rename('n_cells').reset_index())


In [ ]:
started = time.time()
settings = dict(context=None, context_mode='none', fix_phase=False, k_beta=2.0,
                noise_model='nb', fix_disp_val='gene', layer=None,
                n_epochs=N_EPOCHS, batch_size=250, learning_rate=0.001,
                n_theta=24, n_theta_post=N_THETA_POST, init_mean=True, kill_amps=False,
                true_phase=None, fixed_cell_phases=None, device='cpu', posterior_cell_chunk=500)
# batch_size=250 divides 9000, avoiding the official trainer's drop_last omission.
cmodel, losses, mad_epochs = cr.warmup_and_train(
    model_data, params_g.copy(), counts=counts, **settings)
assert model_data.shape == (9000,1000)
assert list(cmodel.genes) == list(params_g.index), 'Warm-start dropped/reordered reference genes.'
assert mad_epochs is None
losses = np.asarray(losses, dtype=float)
assert len(losses) == N_EPOCHS and np.isfinite(losses).all()
assert all(torch.isfinite(p).all().item() for p in cmodel.parameters())
pd.DataFrame({'epoch':np.arange(1,len(losses)+1), 'loss':losses}).to_csv(OUTPUT_DIR/'training_loss.csv',index=False)
tail_change = abs(losses[-20:].mean()-losses[-40:-20].mean()) / max(abs(losses[-40:-20].mean()),1e-12)
qc = {'shape':list(model_data.shape), 'reference_genes':len(cmodel.genes),
      'epochs_completed':len(losses), 'loss_finite':True, 'first_loss':float(losses[0]),
      'last_loss':float(losses[-1]), 'tail_relative_change':float(tail_change),
      'tail_stable_heuristic':bool(tail_change < 0.01),
      'official_convergence_flag':None, 'global_convergence_proven':False,
      'elapsed_seconds':time.time()-started}
if not qc['tail_stable_heuristic']:
    warnings.warn('Loss tail not stable at 1% threshold; inspect training_loss.csv before benchmark interpretation.')
print(json.dumps(qc,indent=2))


In [ ]:
result = meta.reset_index(drop=True).copy()
def vector(attr):
    x = np.asarray(getattr(cmodel,attr)).reshape(-1)
    assert x.shape == (len(result),) and np.isfinite(x).all(), attr
    return x
result['pred_rad'] = vector('post_mode_c') % (2*np.pi)
result['pred_hour'] = result.pred_rad * 24/(2*np.pi)
result['wrapped_error_hour'] = ((result.pred_hour-result.T_true+12)%24)-12
result['abs_error_hour'] = result.wrapped_error_hour.abs()
result['posterior_mean_rad'] = vector('post_mean_c') % (2*np.pi)
result['posterior_mean_hour'] = result.posterior_mean_rad * 24/(2*np.pi)
result['posterior_cstd_rad'] = vector('post_std_c')
result['posterior_cstd_hour'] = result.posterior_cstd_rad * 24/(2*np.pi)
result['posterior_cvar'] = vector('post_var_c')
result['log_mle_per_gene'] = vector('mle_c')
result = result[required + ['pred_hour','pred_rad','wrapped_error_hour','abs_error_hour',
    'posterior_mean_rad','posterior_mean_hour','posterior_cstd_rad','posterior_cstd_hour',
    'posterior_cvar','log_mle_per_gene']]
assert result.cell.tolist() == adata.obs_names.tolist()
assert result.cell.is_unique and not result.isna().any().any()
assert ((result.pred_hour>=0)&(result.pred_hour<24)).all()
assert ((result.wrapped_error_hour>=-12)&(result.wrapped_error_hour<12)).all()
assert np.isfinite(result.select_dtypes(include='number')).all().all()
CSV = OUTPUT_DIR / 'scRITMO_cell_phase.csv'
result.to_csv(CSV,index=False)
check = pd.read_csv(CSV)
assert check.cell.tolist() == result.cell.tolist() and check.shape == result.shape

def metrics(frame):
    e=frame.wrapped_error_hour.to_numpy()
    return dict(n_cells=len(e), circular_MAE_hour=float(np.abs(e).mean()),
                circular_MedAE_hour=float(np.median(np.abs(e))), circular_RMSE_hour=float(np.sqrt(np.mean(e**2))))
rows=[dict(group='overall', celltype='ALL', sigma_bio_hour='ALL', **metrics(result))]
for (ct,sigma), g in result.groupby(['celltype','sigma_bio_hour'],observed=True,sort=True):
    rows.append(dict(group='by_CT_sigma',celltype=ct,sigma_bio_hour=sigma,**metrics(g)))
summary=pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR/'scRITMO_phase_summary.csv',index=False)
qc.update(cell_alignment=True, output_has_na=False, prediction_range_hour=[float(result.pred_hour.min()),float(result.pred_hour.max())])
(OUTPUT_DIR/'phase_QC.json').write_text(json.dumps(qc,indent=2))
manifest=dict(official_commit=OFFICIAL_COMMIT, source_repo=str(REPO), source_sha256=SOURCE_SHA256,
              input_h5ad=str(H5AD), input_sha256=sha256(H5AD), settings=settings, seed=SEED,
              estimator='posterior_mode', python=sys.executable,
              versions={k:__import__(k).__version__ for k in ['torch','numpy','pandas','anndata']},
              csv=str(CSV), csv_sha256=sha256(CSV))
(OUTPUT_DIR/'prediction_manifest.json').write_text(json.dumps(manifest,indent=2))
display(summary)
print('R input:', CSV.resolve())
print('Columns:', ','.join(result.columns))
print('R code:')
print('scRITMO_cell_phase <- read.csv(' + json.dumps(str(CSV.resolve())) + ', check.names = FALSE)')
